# **Full-diagram 1-loop** vs simulation — Allen-Cahn $\phi^4$ (d = 1)

This notebook tests the **genuine full-diagram integrator** (`msrjd.integration.spatial.full_integrator`)
against a direct simulation of the stochastic PDE

$$\partial_t\phi = D\,\partial_x^2\phi - \mu\phi - \lambda\phi^3 + \eta,\qquad
  \langle\eta(x,t)\eta(x',t')\rangle = 2T\,\delta(x-x')\delta(t-t').$$

The framework evaluates **every** enumerated Feynman diagram by the *same* integral —
enumerate $\to$ route momenta $\to$ Schwinger/Symanzik loop-momentum integral $\to$ causal-chamber
time integral $\to$ sum — with **no shortcuts** (no Dyson convolution, no mass-shift, no model-specific
formula).  At $\lambda=0$ it is the exact Gaussian correlator $C_0$; at $O(\lambda)$ the 1-loop **tadpole**
self-energy suppresses the variance (the Hartree shift $\mu\to\mu+3\lambda\langle\phi^2\rangle$).

We compare **tree** ($\texttt{max\_ell=0}$) and **tree + 1-loop** ($\texttt{max\_ell=1}$) to the simulated
equal-time correlator $C(x,0)=\langle\phi(0)\phi(x)\rangle$.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import matplotlib.pyplot as plt
from pipeline.compute import compute_cumulants
from pipeline.theory import TheoryBuilder
from models.spatial_field_1d_sim import simulate, equal_time_correlator

mu, D, T, lam = 1.0, 1.0, 1.0, 0.1

def allen_cahn(l):
    return (TheoryBuilder('allen-cahn-1d', n_populations=0)
            .physical_field('phi', spatial_dim=1)
            .parameter('mu', default=mu, domain='positive')
            .parameter('D',  default=D,  domain='positive')
            .parameter('lam', default=l, domain='positive')
            .parameter('T',  default=T,  domain='positive')
            .equation(lhs='(Dt + mu - D*Laplacian)*phi', rhs='-lam*phi^3')
            .set_action_text('phit*((Dt + mu - D*Laplacian)*phi + lam*phi^3) - T*phit^2')
            .boundary('infinite').initial('stationary').build())

## 1. Theory: tree vs tree + 1-loop through `compute_cumulants`

`max_ell=1` runs the full-diagram integrator on every enumerated $O(\lambda)$ diagram (here the
single $\tilde\phi\phi^3$ Hartree tadpole) and adds it to the tree.

In [ ]:
xs = np.linspace(0.0, 6.0, 25)
kw = dict(k=2, external_fields=[('phi', 1), ('phi', 1)], spatial_grid=xs,
          tau_max=1.0, tau_step=1.0, verbose=False, use_cache=False, mf_dae_n_starts=4)
fund = {'mu': mu, 'D': D, 'lam': lam, 'T': T}

tree = compute_cumulants(allen_cahn(lam), max_ell=0, fundamental=fund, **kw)
loop = compute_cumulants(allen_cahn(lam), max_ell=1, fundamental=fund, **kw)
mid = tree['C_tau_x'].shape[0] // 2                  # τ = 0 row
C_tree = np.real(tree['C_tau_x'])[mid]
C_loop = np.real(loop['C_tau_x'])[mid]
print('full-integrator 1-loop info:', {k: loop['spatial_info'].get(k)
      for k in ('full_integrator', 'n_ell1_diagrams', 'n_live_diagrams')})
print('variance C(0,0):  tree = %.4f   tree+1-loop = %.4f' % (C_tree[0], C_loop[0]))

## 2. Simulation of the SPDE

A pseudo-spectral Euler–Maruyama integration of the $\phi^4$ Langevin equation on a periodic ring;
the equal-time correlator is averaged over snapshots after burn-in.

In [ ]:
snaps, x_grid, meta = simulate(L=40.0, N=256, mu=mu, D=D, lam=lam, T=T,
                               n_steps=120000, burn_in=20000, record_every=20, seed=1)
Cx_full = equal_time_correlator(snaps)               # C[m] at separation m·dx, length N
half = len(x_grid) // 2 + 1
xc, Cx = x_grid[:half], Cx_full[:half]               # one period side, x ≥ 0
print('sim variance C(0,0) = %.4f   (tree 0.5, 1-loop %.4f)' % (Cx[0], C_loop[0]))

## 3. Compare: tree, tree + 1-loop, simulation

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].plot(xs, C_tree, '--', lw=2, label='tree  $C_0(x)$', color='C0')
ax[0].plot(xs, C_loop, '-',  lw=2, label='tree + 1-loop (full integrator)', color='C3')
ax[0].plot(xc, Cx, 'o', ms=5, label='simulation', color='k', alpha=0.7)
ax[0].set_xlabel('x'); ax[0].set_ylabel('C(x, 0)'); ax[0].set_xlim(0, 6)
ax[0].set_title('equal-time correlator'); ax[0].legend()

ax[1].plot(xs, C_tree, '--', lw=2, color='C0', label='tree')
ax[1].plot(xs, C_loop, '-',  lw=2, color='C3', label='tree + 1-loop')
ax[1].plot(xc, Cx, 'o', ms=5, color='k', alpha=0.7, label='sim')
ax[1].set_yscale('log'); ax[1].set_xlabel('x'); ax[1].set_ylabel('C(x, 0)')
ax[1].set_xlim(0, 6); ax[1].set_title('log scale'); ax[1].legend()
plt.tight_layout(); plt.show()

# how much closer does the 1-loop get to the sim at x=0 (the variance)?
sim0 = Cx[0]
print('|sim − tree|      = %.4f' % abs(sim0 - C_tree[0]))
print('|sim − (1-loop)|  = %.4f   <-- the loop correction moves theory toward the sim'
      % abs(sim0 - C_loop[0]))

## Summary

The **full-diagram integrator** computes the 1-loop correlator for a simple $\phi^4$ theory with no
model-specific input: every diagram is the same `enumerate → Symanzik → causal chambers` integral.
The 1-loop tadpole **suppresses** the variance below the tree value $T/2\sqrt{\mu D}=0.5$, moving the
theory toward the simulated $\langle\phi^2\rangle$.

Knobs to explore: `lam` (coupling — the 1-loop shift scales $\propto\lambda$), `mu`, `D`, the grid `xs`,
and `spatial_dim` in the theory (the integrator is general in $d$; $d\ge2$ tadpoles are UV-sensitive and
set by the Schwinger cutoff).  Derivative/$\nabla$ vertices are future work.